# Notebook 6 — Glued Brownian growth-rate sweep

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from front_runner import compile_c_code, run_front_simulations
from front_analysis import (
    read_front_file,
    get_run_files,
    fit_front_speeds_for_front_data,
    side_speed_long_table,
    style_axis,
)


project_folder = Path('.').resolve()

c_file = project_folder / 'abm_front_propagation_glued.c'
exe_file = project_folder / 'abm_front_glued_B50_growth_sweep.exe'

param_base = 'params_brownian_growth_sweep_glued_B50'
param_file = project_folder / f'{param_base}.txt'
run_table_file = project_folder / f'{param_base}_runs.csv'

data_folder = project_folder / 'data' / param_base
figures_folder = project_folder / 'figures' / param_base
summary_folder = project_folder / 'analysis_summaries' / param_base

for folder in [data_folder, figures_folder, summary_folder]:
    folder.mkdir(parents=True, exist_ok=True)

compile_first = False
run_simulations = False
skip_existing = True
selected_run_ids = None     # Example: [0, 1, 2] for testing only a few runs
max_parallel = 26

print('Project folder:', project_folder)
print('C file:', c_file)
print('Executable:', exe_file)
print('Parameter file:', param_file)
print('Run table:', run_table_file)
print('Data folder:', data_folder)
print('Figures folder:', figures_folder)

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Sweep parameters

In [ ]:
Lx = 80.0
T = 600.0

Ly_values = np.array([2.0], dtype=float)
R_inter_values = np.array([0.05], dtype=float)

Dr = 0.008

r_values = np.array([0.05, 0.10, 0.20, 0.35, 0.50, 0.70], dtype=float)
q0 = 0.15
p0_values = q0 + r_values

if np.any(p0_values > 1.0):
    raise ValueError('Some p0 values exceed 1. Reduce r_values or q0.')

seed_values = np.array([69069, 1234567, 999999], dtype=int)

isolation_buffer_factor = 50.0

v0 = 0.0
Dtheta = 0.002

Ns_fixed = 50
rho0 = 750.0
rho_sat = -1.0

dt = 0.001
warmup_T = 200.0

front_per_step = int(round(0.2 / dt))
save_per_step = int(round(10.0 / dt))
rho_profile_every_front = 10

nbins_x = 1600

def dt_max_for_R_Dr(R, Dr):
    return (0.1 * float(R))**2 / (2.0 * float(Dr))

def v_fkpp_for_r(r):
    return 2.0 * np.sqrt(float(Dr) * float(r))

dt_check_rows = []
for R in R_inter_values:
    dt_check_rows.append({
        'R_inter': float(R),
        'Dr': float(Dr),
        'dt_max': dt_max_for_R_Dr(R, Dr),
        'current_dt': dt,
        'current_dt_is_safe': dt <= dt_max_for_R_Dr(R, Dr),
    })
dt_check = pd.DataFrame(dt_check_rows)

print('Total planned runs:', len(Ly_values) * len(R_inter_values) * len(r_values) * len(seed_values))
print('Isolation buffer factor:', isolation_buffer_factor)
print('Dr:', Dr)
print('q0:', q0)
print('r values:', r_values)
print('p0 values:', p0_values)
display(dt_check)

if not dt_check['current_dt_is_safe'].all():
    raise ValueError('Current dt violates the Brownian displacement condition for at least one R_inter.')

## 2. Write the glued parameter file

In [ ]:
base = dict(
    q0=q0,
    Ns=Ns_fixed,
    rho0=rho0,
    v0=v0,
    Dr=Dr,
    Dtheta=Dtheta,
    dt=dt,
    T=T,
    Lx=Lx,
    warmup_T=warmup_T,
    rho_sat=rho_sat,
    nbins_x=nbins_x,
    threshold_frac1=0.2,
    threshold_frac2=0.5,
    threshold_frac3=0.8,
    save_per_step=save_per_step,
    front_per_step=front_per_step,
    rho_profile_every_front=rho_profile_every_front,
    isolation_buffer_factor=isolation_buffer_factor,
)

param_order = [
    'run_id', 'seed',
    'p0', 'q0', 'Ns', 'R_inter', 'rho0',
    'save_per_step', 'front_per_step', 'rho_profile_every_front',
    'v0', 'Dr', 'Dtheta', 'dt', 'T',
    'Lx', 'Ly', 'x_init_min', 'x_init_max', 'warmup_T',
    'rho_sat', 'nbins_x', 'threshold_frac1', 'threshold_frac2', 'threshold_frac3',
    'isolation_buffer_factor',
]

runs = []
run_id = 0

for Ly in Ly_values:
    for R in R_inter_values:
        for r_edge in r_values:
            for seed in seed_values:
                run = dict(base)
                run.update(
                    run_id=run_id,
                    seed=int(seed),
                    p0=float(q0 + r_edge),
                    R_inter=float(R),
                    Ly=float(Ly),
                    x_init_min=0.5 * Lx - 0.25,
                    x_init_max=0.5 * Lx + 0.25,
                )
                run['r_edge'] = run['p0'] - run['q0']
                run['v_fkpp'] = v_fkpp_for_r(run['r_edge'])
                run['code_type'] = 'glued'
                runs.append(run)
                run_id += 1

run_table = pd.DataFrame(runs)
run_table.to_csv(run_table_file, index=False)

with open(param_file, 'w') as f:
    f.write('# Glued Brownian growth-rate sweep with B = 30 R_inter\n')
    f.write('# Sweep r = p0 - q0 by changing p0 and keeping q0 fixed\n')
    f.write('# Uses glued C code and saves front files for velocity analysis\n')
    f.write(f'# Dr = {Dr:.16g}\n')
    f.write(f'# dt = {dt:.16g}\n')
    f.write('# condition checked: sqrt(2 Dr dt) <= 0.1 R_inter\n\n')
    for run in runs:
        f.write('[run]\n')
        for key in param_order:
            f.write(f'{key} = {run[key]}\n')
        f.write('\n')

print('Wrote:', param_file)
print('Wrote:', run_table_file)
display(run_table.head(12))

## 3. Compile and run

In [ ]:
if compile_first:
    compile_c_code(
        c_file=c_file,
        exe_file=exe_file,
        flags=['-O2', '-std=c99', '-Wall', '-Wextra'],
    )
else:
    print('compile_first = False')

In [ ]:
if run_simulations:
    results = run_front_simulations(
        exe_file=exe_file,
        param_file=param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing,
        required_output='front',
    )
    display(pd.DataFrame(results))
else:
    print('run_simulations = False')

## 4. Fit front speeds

In [ ]:
main_method = 'th_2'
main_fit_window = (0.70, 0.95)

fit_windows = [
    (0.50, 0.90),
    (0.60, 0.90),
    (0.60, 0.95),
    (0.70, 0.90),
    (0.70, 0.95),
    (0.80, 0.95),
]

def window_label(a, b):
    return f'{a:.2f}-{b:.2f}'

def clean_front_data(front_data, exclude_boundary_hit_rows=True):
    n = len(front_data['time'])
    mask = np.isfinite(np.asarray(front_data['time'], dtype=float))
    if exclude_boundary_hit_rows and 'hit_boundary' in front_data:
        mask &= np.asarray(front_data['hit_boundary'], dtype=float) < 0.5

    out = {}
    for key, value in front_data.items():
        arr = np.asarray(value)
        if arr.shape == (n,):
            out[key] = arr[mask]
        else:
            out[key] = value
    return out

def read_front_file_safe(front_file):
    try:
        return read_front_file(front_file)
    except TypeError as err:
        if 'delim_whitespace' not in str(err):
            raise

        front_file = Path(front_file)
        header_cols = None
        metadata = {}
        with open(front_file, 'r') as f:
            for line in f:
                if line.startswith('# step '):
                    header_cols = line[1:].strip().split()
                    break
        if header_cols is None:
            raise ValueError(f'Could not find front-file header in {front_file}')

        df = pd.read_csv(
            front_file,
            comment='#',
            sep=r'\s+',
            header=None,
            names=header_cols,
        )
        front_data = {col: df[col].to_numpy() for col in df.columns}
        return metadata, front_data

def fit_runs_for_window(run_table, f0, f1):
    fit_rows = []
    missing = []

    for _, run in run_table.iterrows():
        rid = int(run['run_id'])
        _, front_file, _ = get_run_files(data_folder, param_base, rid)
        front_file = Path(front_file)

        if not front_file.exists():
            missing.append(rid)
            continue

        _, front_data = read_front_file_safe(front_file)
        front_data = clean_front_data(front_data, exclude_boundary_hit_rows=True)

        fit = fit_front_speeds_for_front_data(
            front_data,
            methods=threshold_methods,
            fit_start_fraction=f0,
            fit_end_fraction=f1,
        )

        row = run.to_dict()
        row.update(fit)
        row['fit_window'] = window_label(f0, f1)
        row['fit_start_fraction'] = f0
        row['fit_end_fraction'] = f1
        fit_rows.append(row)

    return pd.DataFrame(fit_rows), missing

run_table = pd.read_csv(run_table_file)

run_table = run_table.loc[
    run_table["r_edge"] >= 0.05 - 1e-12
].copy()

run_table.reset_index(drop=True, inplace=True)

fit_df, missing = fit_runs_for_window(run_table, *main_fit_window)

print(f'Main window {window_label(*main_fit_window)}: fitted {len(fit_df)} / {len(run_table)} runs')
if missing:
    print('Missing front files:', missing[:30], '...' if len(missing) > 30 else '')

fit_df.to_csv(summary_folder / 'fit_df_main_window.csv', index=False)
display(fit_df.head())

## 5. Run-level speed summary

In [ ]:
def make_run_level_speed_table(fit_df, group_cols):
    keep_cols = ['run_id'] + list(group_cols) + ['v_fkpp']
    keep_cols = [c for c in keep_cols if c in fit_df.columns]

    speed_long = side_speed_long_table(
        fit_df,
        method=main_method,
        keep_columns=keep_cols,
    )

    speed_long = speed_long.replace([np.inf, -np.inf], np.nan)
    speed_long = speed_long.dropna(subset=['speed', 'v_fkpp'])
    speed_long['speed_over_fkpp'] = speed_long['speed'] / speed_long['v_fkpp']

    run_cols = ['run_id'] + list(group_cols)
    run_level = (
        speed_long
        .groupby(run_cols, as_index=False)
        .agg(
            run_speed=('speed', 'mean'),
            run_speed_over_fkpp=('speed_over_fkpp', 'mean'),
            left_right_std=('speed_over_fkpp', 'std'),
            n_front_sides=('speed_over_fkpp', 'count'),
            v_fkpp=('v_fkpp', 'first'),
        )
    )

    return run_level

def summarize_run_level(run_level, group_cols):
    summary = (
        run_level
        .groupby(group_cols, as_index=False)
        .agg(
            n_runs=('run_speed', 'count'),
            mean_speed=('run_speed', 'mean'),
            speed_std=('run_speed', 'std'),
            mean_speed_over_fkpp=('run_speed_over_fkpp', 'mean'),
            speed_over_fkpp_std=('run_speed_over_fkpp', 'std'),
            v_fkpp=('v_fkpp', 'first'),
        )
    )

    summary['speed_sem'] = summary['speed_std'] / np.sqrt(summary['n_runs'])
    summary['speed_over_fkpp_sem'] = summary['speed_over_fkpp_std'] / np.sqrt(summary['n_runs'])
    summary['delta_from_fkpp'] = summary['mean_speed_over_fkpp'] - 1.0
    summary['abs_delta_from_fkpp'] = np.abs(summary['delta_from_fkpp'])
    summary['v2_measured'] = summary['mean_speed']**2
    summary['v2_fkpp'] = summary['v_fkpp']**2

    return summary

group_cols = ['Ly', 'R_inter', 'Dr', 'r_edge']
run_level_main = make_run_level_speed_table(fit_df, group_cols)
summary = summarize_run_level(run_level_main, group_cols)

run_level_main.to_csv(summary_folder / 'run_level_speeds_main_window.csv', index=False)
summary.to_csv(summary_folder / 'summary_main_window.csv', index=False)

display(summary.sort_values(['Ly', 'R_inter', 'r_edge']).head(30))

## 6. Normal plot: measured speed versus growth rate

In [ ]:
if len(summary) == 0:
    print('No summary to plot yet.')
else:
    ncols = len(Ly_values)
    fig, axes = plt.subplots(1, ncols, figsize=(5.2 * ncols, 4.2), sharey=True)
    if ncols == 1:
        axes = [axes]

    r_line = np.linspace(r_values.min(), r_values.max(), 300)
    v_fkpp_line = 2.0 * np.sqrt(Dr * r_line)

    for ax, Ly in zip(axes, Ly_values):
        sub_Ly = summary[np.isclose(summary['Ly'], Ly)]

        for R in R_inter_values:
            sub = sub_Ly[np.isclose(sub_Ly['R_inter'], R)].sort_values('r_edge')
            if len(sub) == 0:
                continue
            ax.errorbar(
                sub['r_edge'],
                sub['mean_speed'],
                yerr=sub['speed_sem'].fillna(0.0),
                marker='o',
                linestyle='-',
                capsize=3,
                linewidth=1.5,
                label=fr'$R_{{inter}}={R:g}$',
            )

        ax.plot(r_line, v_fkpp_line, '--', linewidth=1.7, label='FKPP')
        ax.set_title(fr'$L_y={Ly:g}$')
        ax.set_xlabel(r'$r=p_0-q_0$')
        ax.grid(alpha=0.25)

    axes[0].set_ylabel(r'$v_{sim}$')
    axes[-1].legend(fontsize=9, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(fr'Glued Brownian growth sweep, $D_r={Dr:g}$, $B={isolation_buffer_factor:g}R_{{inter}}$, fit {window_label(*main_fit_window)}', y=1.03)
    fig.tight_layout()

    save_path = figures_folder / 'glued_growth_velocity_vs_r_normal_scale.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))

ax.plot(
    summary['r_edge'],
    summary['v2_measured'],
    marker='o',
    linestyle='',
    label=r'measured $v^2$'
)

r_line = np.linspace(summary['r_edge'].min(), summary['r_edge'].max(), 200)
Dr_ref = Dr
ax.plot(
    r_line,
    4.0 * Dr_ref * r_line,
    '--',
    label=r'FKPP $4D_r(p_0-q_0)$'
)

valid = np.isfinite(summary['r_edge']) & np.isfinite(summary['v2_measured'])

if valid.sum() >= 2:
    x_fit = summary.loc[valid, 'r_edge'].to_numpy()
    y_fit = summary.loc[valid, 'v2_measured'].to_numpy()

    slope, intercept = np.polyfit(x_fit, y_fit, 1)

    y_pred = slope * x_fit + intercept
    residuals = y_fit - y_pred

    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)

    ax.plot(
        r_line,
        slope * r_line + intercept,
        ':',
        label=fr'measured fit: slope={slope:.4g}'
    )

    print(f'Measured v^2 slope          = {slope:.6g}')
    print(f'FKPP v^2 slope              = {4.0 * Dr_ref:.6g}')
    print(f'Intercept                   = {intercept:.6g}')
    print(f'MSE of measured linear fit  = {mse:.6g}')
    print(f'RMSE of measured linear fit = {rmse:.6g}')

ax.set_xlabel(r'$p_0 - q_0$')
ax.set_ylabel(r'$v^2$')
ax.set_title('Brownian FKPP test: squared speed versus growth rate')
ax.grid(False)
ax.legend()
fig.tight_layout()

save_path = figures_folder / 'brownian_growth_sweep_v2_vs_growth.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 7. Normalized plot: $v_{sim}/v_{FKPP}$

In [ ]:
if len(summary) == 0:
    print('No summary to plot yet.')
else:
    ncols = len(Ly_values)
    fig, axes = plt.subplots(1, ncols, figsize=(5.2 * ncols, 4.2), sharey=True)
    if ncols == 1:
        axes = [axes]

    for ax, Ly in zip(axes, Ly_values):
        sub_Ly = summary[np.isclose(summary['Ly'], Ly)]

        for R in R_inter_values:
            sub = sub_Ly[np.isclose(sub_Ly['R_inter'], R)].sort_values('r_edge')
            if len(sub) == 0:
                continue
            ax.errorbar(
                sub['r_edge'],
                sub['mean_speed_over_fkpp'],
                yerr=sub['speed_over_fkpp_sem'].fillna(0.0),
                marker='o',
                linestyle='-',
                capsize=3,
                linewidth=1.5,
                label=fr'$R_{{inter}}={R:g}$',
            )

        ax.axhline(1.0, linestyle='--', linewidth=1.2)
        ax.set_title(fr'$L_y={Ly:g}$')
        ax.set_xlabel(r'$r=p_0-q_0$')
        ax.grid(alpha=0.25)

    axes[0].set_ylabel(r'$v_{sim}/v_{FKPP}$')
    axes[-1].legend(fontsize=9, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(fr'Glued Brownian growth FKPP ratio, $D_r={Dr:g}$, $B={isolation_buffer_factor:g}R_{{inter}}$, fit {window_label(*main_fit_window)}', y=1.03)
    fig.tight_layout()

    save_path = figures_folder / 'glued_growth_v_over_fkpp_vs_r.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 8. Closest-to-FKPP table

In [ ]:
if len(summary) == 0:
    print('No summary to rank yet.')
else:
    geometry_ranking = (
        summary
        .groupby(['Ly', 'R_inter'], as_index=False)
        .agg(
            n_r=('r_edge', 'count'),
            mean_abs_error_to_fkpp=('abs_delta_from_fkpp', 'mean'),
            max_abs_error_to_fkpp=('abs_delta_from_fkpp', 'max'),
            mean_signed_error_to_fkpp=('delta_from_fkpp', 'mean'),
            mean_v_over_fkpp=('mean_speed_over_fkpp', 'mean'),
        )
        .sort_values(['mean_abs_error_to_fkpp', 'max_abs_error_to_fkpp'])
        .reset_index(drop=True)
    )
    geometry_ranking.insert(0, 'rank_closest_to_fkpp', np.arange(1, len(geometry_ranking) + 1))

    geometry_ranking.to_csv(summary_folder / 'geometry_ranking_closest_to_fkpp_growth_sweep.csv', index=False)
    display(geometry_ranking)

## 9. Fit-window sensitivity / stationarity check

In [ ]:
all_fit_rows = []
all_window_summaries = []

for f0, f1 in fit_windows:
    label = window_label(f0, f1)
    fit_df_w, missing_w = fit_runs_for_window(run_table, f0, f1)
    print(f'Window {label}: fitted {len(fit_df_w)} / {len(run_table)} runs')
    if missing_w:
        print('  Missing examples:', missing_w[:10], '...' if len(missing_w) > 10 else '')

    if len(fit_df_w) == 0:
        continue

    run_level_w = make_run_level_speed_table(fit_df_w, group_cols)
    summary_w = summarize_run_level(run_level_w, group_cols)
    summary_w['fit_window'] = label
    summary_w['fit_start_fraction'] = f0
    summary_w['fit_end_fraction'] = f1

    fit_df_w['fit_window'] = label
    all_fit_rows.append(fit_df_w)
    all_window_summaries.append(summary_w)

if len(all_window_summaries) > 0:
    window_summary = pd.concat(all_window_summaries, ignore_index=True)
    all_window_fits = pd.concat(all_fit_rows, ignore_index=True)
else:
    window_summary = pd.DataFrame()
    all_window_fits = pd.DataFrame()

window_summary.to_csv(summary_folder / 'fit_window_summary.csv', index=False)
all_window_fits.to_csv(summary_folder / 'fit_window_all_fits.csv', index=False)

display(window_summary.head(20))

## 10. Fit-window sensitivity plot: $v_{sim}/v_{FKPP}$

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary to plot yet.')
else:
    nrows = len(fit_windows)
    ncols = len(Ly_values)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.2 * ncols, 2.9 * nrows),
        sharex=True,
        sharey=True,
    )

    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = np.array([axes])
    elif ncols == 1:
        axes = np.array([[ax] for ax in axes])

    for row_idx, (f0, f1) in enumerate(fit_windows):
        label = window_label(f0, f1)

        for col_idx, Ly in enumerate(Ly_values):
            ax = axes[row_idx, col_idx]
            sub_Ly = window_summary[
                (window_summary['fit_window'] == label) &
                np.isclose(window_summary['Ly'], Ly)
            ]

            for R in R_inter_values:
                sub = sub_Ly[np.isclose(sub_Ly['R_inter'], R)].sort_values('r_edge')
                if len(sub) == 0:
                    continue
                ax.errorbar(
                    sub['r_edge'],
                    sub['mean_speed_over_fkpp'],
                    yerr=sub['speed_over_fkpp_sem'].fillna(0.0),
                    marker='o',
                    linestyle='-',
                    capsize=3,
                    linewidth=1.2,
                    label=fr'$R={R:g}$',
                )

            ax.axhline(1.0, linestyle='--', linewidth=1.0)
            ax.grid(alpha=0.25)
            if row_idx == 0:
                ax.set_title(fr'$L_y={Ly:g}$')
            if col_idx == 0:
                ax.set_ylabel(f'fit {label}\n' + r'$v_{sim}/v_{FKPP}$')
            if row_idx == nrows - 1:
                ax.set_xlabel(r'$r=p_0-q_0$')

    axes[0, -1].legend(fontsize=9, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(fr'Fit-window sensitivity, glued $B={isolation_buffer_factor:g}R_{{inter}}$, $D_r={Dr:g}$', y=1.01)
    fig.tight_layout()

    save_path = figures_folder / 'glued_growth_v_over_fkpp_fit_window_sensitivity.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 11. Fit-window spread summary

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary yet.')
else:
    window_spread = (
        window_summary
        .groupby(['Ly', 'R_inter', 'Dr', 'r_edge'], as_index=False)
        .agg(
            min_v_over_fkpp=('mean_speed_over_fkpp', 'min'),
            max_v_over_fkpp=('mean_speed_over_fkpp', 'max'),
            mean_v_over_fkpp=('mean_speed_over_fkpp', 'mean'),
            std_v_over_fkpp_across_windows=('mean_speed_over_fkpp', 'std'),
            n_windows=('mean_speed_over_fkpp', 'count'),
        )
    )
    window_spread['range_v_over_fkpp_across_windows'] = (
        window_spread['max_v_over_fkpp'] - window_spread['min_v_over_fkpp']
    )
    window_spread['abs_mean_error_to_fkpp'] = np.abs(window_spread['mean_v_over_fkpp'] - 1.0)

    window_spread = window_spread.sort_values(
        ['range_v_over_fkpp_across_windows', 'abs_mean_error_to_fkpp']
    ).reset_index(drop=True)

    window_spread.to_csv(summary_folder / 'fit_window_spread_stationarity_ranking.csv', index=False)
    display(window_spread.head(40))

    geometry_stationarity = (
        window_spread
        .groupby(['Ly', 'R_inter'], as_index=False)
        .agg(
            mean_window_range=('range_v_over_fkpp_across_windows', 'mean'),
            max_window_range=('range_v_over_fkpp_across_windows', 'max'),
            mean_abs_error_to_fkpp=('abs_mean_error_to_fkpp', 'mean'),
        )
        .sort_values(['mean_window_range', 'mean_abs_error_to_fkpp'])
        .reset_index(drop=True)
    )
    geometry_stationarity.insert(0, 'rank_most_stationary', np.arange(1, len(geometry_stationarity) + 1))

    geometry_stationarity.to_csv(summary_folder / 'geometry_stationarity_ranking_growth_sweep.csv', index=False)
    display(geometry_stationarity)

## 12. Compact final table

In [ ]:
if len(summary) == 0 or len(window_summary) == 0:
    print('Need both main summary and fit-window summary.')
else:
    if 'geometry_ranking' not in globals():
        geometry_ranking = pd.DataFrame()
    if 'geometry_stationarity' not in globals():
        geometry_stationarity = pd.DataFrame()

    if len(geometry_ranking) > 0 and len(geometry_stationarity) > 0:
        final_geometry_table = geometry_ranking.merge(
            geometry_stationarity,
            on=['Ly', 'R_inter'],
            how='left',
            suffixes=('_fkpp', '_stationarity'),
        )

        final_geometry_table = final_geometry_table[[
            'Ly', 'R_inter',
            'rank_closest_to_fkpp',
            'mean_abs_error_to_fkpp_fkpp',
            'max_abs_error_to_fkpp',
            'mean_signed_error_to_fkpp',
            'mean_v_over_fkpp',
            'rank_most_stationary',
            'mean_window_range',
            'max_window_range',
        ]].sort_values(['rank_closest_to_fkpp', 'rank_most_stationary'])

        final_geometry_table.to_csv(summary_folder / 'final_geometry_choice_table_growth_sweep.csv', index=False)
        display(final_geometry_table)
    else:
        print('Geometry ranking was not generated.')

    concise = summary[[
        'Ly', 'R_inter', 'Dr', 'r_edge', 'n_runs',
        'mean_speed', 'speed_sem', 'v_fkpp',
        'mean_speed_over_fkpp', 'speed_over_fkpp_sem',
        'delta_from_fkpp', 'abs_delta_from_fkpp',
    ]].sort_values(['Ly', 'R_inter', 'r_edge'])
    concise.to_csv(summary_folder / 'final_growth_sweep_summary.csv', index=False)
    display(concise)

## Threshold comparison

In [ ]:
# ==========================================================
# Threshold comparison summary
# Compare th_1, th_2, th_3 as v_front(r)
# ==========================================================

threshold_methods = ["th_1", "th_2", "th_3"]

threshold_label_map = {
    "th_1": rf"${base['threshold_frac1']:g}\rho_{{\mathrm{{sat}}}}$",
    "th_2": rf"${base['threshold_frac2']:g}\rho_{{\mathrm{{sat}}}}$",
    "th_3": rf"${base['threshold_frac3']:g}\rho_{{\mathrm{{sat}}}}$",
}

threshold_rows = []

keep_cols = [
    "run_id",
    "seed",
    "Ly",
    "R_inter",
    "Dr",
    "r_edge",
    "v_fkpp",
]
keep_cols = [c for c in keep_cols if c in fit_df.columns]

for method in threshold_methods:
    speed_long = side_speed_long_table(
        fit_df,
        method=method,
        keep_columns=keep_cols,
    )

    if len(speed_long) == 0:
        print(
            f"No data found for {method}. "
            "Check that fit_df contains the corresponding "
            "left/right speed columns."
        )
        continue

    speed_long = speed_long.replace([np.inf, -np.inf], np.nan)
    speed_long = speed_long.dropna(subset=["speed"]).copy()

    threshold_rows.append(speed_long)

if len(threshold_rows) == 0:
    threshold_long = pd.DataFrame()
    threshold_run_level = pd.DataFrame()
    threshold_summary = pd.DataFrame()
    print("No threshold data found.")

else:
    threshold_long = pd.concat(
        threshold_rows,
        ignore_index=True,
    )

    group_cols = [
        "Ly",
        "R_inter",
        "Dr",
        "r_edge",
        "method",
    ]

    run_index = ["run_id"] + group_cols

    # ----------------------------------------------------------
    # Average the left and right fronts within each realization
    # ----------------------------------------------------------

    threshold_run_level = (
        threshold_long
        .pivot_table(
            index=run_index,
            columns="side",
            values="speed",
            aggfunc="mean",
        )
        .reset_index()
    )

    threshold_run_level.columns.name = None

    if (
        "left" not in threshold_run_level.columns
        or "right" not in threshold_run_level.columns
    ):
        threshold_run_level = pd.DataFrame()
        threshold_summary = pd.DataFrame()

        print(
            "Both left and right front speeds are required "
            "for the threshold comparison."
        )

    else:
        # Use only realizations for which both fronts are available.
        threshold_run_level = threshold_run_level.replace(
            [np.inf, -np.inf],
            np.nan,
        )

        threshold_run_level = threshold_run_level.dropna(
            subset=["left", "right"]
        ).copy()

        threshold_run_level["run_speed"] = 0.5 * (
            threshold_run_level["left"]
            + threshold_run_level["right"]
        )

        # ------------------------------------------------------
        # Preserve the FKPP reference for each realization
        # ------------------------------------------------------

        if "v_fkpp" in threshold_long.columns:
            run_metadata = (
                threshold_long
                .groupby(run_index, as_index=False)
                .agg(v_fkpp=("v_fkpp", "first"))
            )

            threshold_run_level = threshold_run_level.merge(
                run_metadata,
                on=run_index,
                how="left",
            )

            threshold_run_level["run_speed_over_fkpp"] = (
                threshold_run_level["run_speed"]
                / threshold_run_level["v_fkpp"]
            )

        # ------------------------------------------------------
        # Statistics across independent realizations
        # ------------------------------------------------------

        threshold_summary = (
            threshold_run_level
            .groupby(group_cols, as_index=False)
            .agg(
                n_runs=("run_speed", "count"),
                mean_speed=("run_speed", "mean"),
                speed_std=("run_speed", "std"),
            )
        )

        threshold_summary["speed_sem"] = (
            threshold_summary["speed_std"]
            / np.sqrt(threshold_summary["n_runs"])
        )

        if "run_speed_over_fkpp" in threshold_run_level.columns:
            ratio_summary = (
                threshold_run_level
                .groupby(group_cols, as_index=False)
                .agg(
                    mean_speed_over_fkpp=(
                        "run_speed_over_fkpp",
                        "mean",
                    ),
                    speed_over_fkpp_std=(
                        "run_speed_over_fkpp",
                        "std",
                    ),
                )
            )

            threshold_summary = threshold_summary.merge(
                ratio_summary,
                on=group_cols,
                how="left",
            )

            threshold_summary["speed_over_fkpp_sem"] = (
                threshold_summary["speed_over_fkpp_std"]
                / np.sqrt(threshold_summary["n_runs"])
            )

        display(
            threshold_summary.sort_values(
                ["Ly", "R_inter", "method", "r_edge"]
            )
        )

In [ ]:
# ==========================================================
# Plot: v_front vs growth rate for the three thresholds
# ==========================================================

if len(threshold_summary) == 0:
    print("No threshold summary to plot yet.")
else:
    plot_Ly = float(Ly_values[0])
    plot_R = float(R_inter_values[0])
    plot_Dr = float(Dr)

    plot_df = threshold_summary[
        np.isclose(threshold_summary["Ly"], plot_Ly)
        & np.isclose(threshold_summary["R_inter"], plot_R)
        & np.isclose(threshold_summary["Dr"], plot_Dr)
    ].copy()

    fig, ax = plt.subplots(figsize=(7.0, 4.5))

    for method in threshold_methods:
        sub = (
            plot_df[plot_df["method"] == method]
            .sort_values("r_edge")
        )

        if len(sub) == 0:
            continue

        ax.errorbar(
            sub["r_edge"],
            sub["mean_speed"],
            yerr=sub["speed_sem"].fillna(0.0),
            marker="o",
            linestyle="-",
            capsize=3,
            linewidth=1.5,
            label=threshold_label_map.get(method, method),
        )

    r_line = np.linspace(plot_df["r_edge"].min(), plot_df["r_edge"].max(), 300)
    ax.plot(
        r_line,
        2.0 * np.sqrt(plot_Dr * r_line),
        "--",
        linewidth=1.5,
        label="FKPP",
    )

    ax.set_xlabel(r"$r = p_0 - q_0$")
    ax.set_ylabel(r"$v_{\mathrm{front}}$")
    ax.set_title(
        fr"Threshold comparison, $L_y={plot_Ly:g}$, "
        fr"$R={plot_R:g}$, $D_r={plot_Dr:g}$"
    )

    ax.legend(frameon=False)
    style_axis(ax, n_ticks=4, grid=False)

    fig.tight_layout()
    save_path = figures_folder / "threshold_comparison_vfront_vs_growth_rate.png"
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()

## Thesis plots

In [ ]:
from matplotlib.ticker import MaxNLocator

R_colors = {
    0.05: "tab:blue",
    0.08: "tab:orange",
    0.10: "tab:green",
}

color_fkpp = "tab:red"
color_fit = "0.55"   # grey

def get_R_color(R):
    for R_key, color in R_colors.items():
        if np.isclose(R, R_key):
            return color
    return None

In [ ]:
Ly_target = 2.0
R_target = 0.05

if len(summary) == 0:
    print("No summary to plot yet.")
else:
    sub = summary[
        np.isclose(summary["Ly"], Ly_target)
        & np.isclose(summary["R_inter"], R_target)
    ].sort_values("r_edge").copy()

    if len(sub) == 0:
        print(f"No data found for Ly={Ly_target} and R={R_target}")
    else:
        x = sub["r_edge"].to_numpy()
        y = sub["mean_speed"].to_numpy()
        yerr = sub["speed_sem"].fillna(0.0).to_numpy()

        y2 = y**2

        y2err = 2.0 * y * yerr

        r_line = np.linspace(x.min(), x.max(), 300)
        v_fkpp_line = 2.0 * np.sqrt(Dr * r_line)
        v_fkpp2_line = 4.0 * Dr * r_line

        fit_slope, fit_intercept = np.polyfit(x, y2, deg=1)
        fit_line = fit_slope * r_line + fit_intercept

        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))

        axes[0].errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            linestyle="-",
            capsize=3,
            linewidth=1.5,
            color=get_R_color(R_target),
            label=fr"$R={R_target:g}$",
        )

        axes[0].plot(
            r_line,
            v_fkpp_line,
            "--",
            linewidth=1.7,
            color=color_fkpp,
            label="FKPP",
        )

        axes[0].set_xlabel(r"$r=p_0-q_0$")
        axes[0].set_ylabel(r"$v_{\mathrm{front}}$")
        axes[0].grid(False)
        axes[0].legend(fontsize=14, frameon=False)

        axes[0].xaxis.set_major_locator(MaxNLocator(nbins=4))
        axes[0].yaxis.set_major_locator(MaxNLocator(nbins=4))

        axes[1].errorbar(
            x,
            y2,
            yerr=y2err,
            marker="o",
            linestyle="none",
            capsize=3,
            linewidth=1.5,
            color=get_R_color(R_target),
            label=fr"$R={R_target:g}$",
        )

        axes[1].plot(
            r_line,
            fit_line,
            "-",
            linewidth=1.7,
            color=color_fit,
            label="Linear fit",
        )

        axes[1].plot(
            r_line,
            v_fkpp2_line,
            "--",
            linewidth=1.7,
            color=color_fkpp,
            label="FKPP",
        )

        axes[1].set_xlabel(r"$r=p_0-q_0$")
        axes[1].set_ylabel(r"$v_{\mathrm{front}}^2$")
        axes[1].grid(False)
        axes[1].legend(fontsize=14, frameon=False)

        axes[1].xaxis.set_major_locator(MaxNLocator(nbins=4))
        axes[1].yaxis.set_major_locator(MaxNLocator(nbins=4))

        fig.tight_layout()

        save_path = figures_folder / (
            f"brownian_growth_velocity_scaling_"
            f"Ly{Ly_target:g}_R{str(R_target).replace('.', 'p')}.pdf"
        )
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("Saved:", save_path)

        print(f"Linear fit: v_front^2 = {fit_slope:.6g} r + {fit_intercept:.6g}")
        print(f"FKPP slope: 4Dr = {4.0 * Dr:.6g}")

        plt.show()

In [ ]:
Ly_target = 2.0
R_target = 0.05

if len(summary) == 0:
    print("No summary to plot yet.")
else:
    sub = summary[
        np.isclose(summary["Ly"], Ly_target)
        & np.isclose(summary["R_inter"], R_target)
    ].sort_values("r_edge").copy()

    if len(sub) == 0:
        print(f"No data found for Ly={Ly_target} and R={R_target}")
    else:
        fig, ax = plt.subplots(figsize=(5.5, 4.2))

        ax.errorbar(
            sub["r_edge"],
            sub["mean_speed_over_fkpp"],
            yerr=sub["speed_over_fkpp_sem"].fillna(0.0),
            marker="o",
            linestyle="-",
            capsize=3,
            linewidth=1.5,
            color=get_R_color(R_target),
            label=fr"$R={R_target:g}$",
        )

        ax.axhline(
            1.0,
            linestyle="--",
            linewidth=1.7,
            color=color_fkpp,
            label="FKPP",
        )

        ax.set_xlabel(r"$r=p_0-q_0$")
        ax.set_ylabel(r"$v_{\mathrm{front}}/v_{\mathrm{FKPP}}$")
        ax.grid(False)
        ax.legend(fontsize=14, frameon=False)

        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

        fig.tight_layout()

        save_path = figures_folder / (
            f"brownian_growth_v_over_fkpp_"
            f"Ly{Ly_target:g}_R{str(R_target).replace('.', 'p')}.pdf"
        )
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("Saved:", save_path)

        plt.show()